# 01 — Pipeline walkthrough

This notebook explains the logic of the code without running a long production simulation.

## Phase-space convention

The packed state is

\[
z=(R,P,r_0,r_1,p_0,p_1).
\]

The nuclear variables are \((R,P)\), and the electronic mapping variables are \((r_0,r_1,p_0,p_1)\).

In [ ]:
from gp_mqcld.Models import TullyModel, TullyParams
from gp_mqcld.Mint import PBMEMIntDynamics, PBMEMIntParams, pack_z, unpack_z
from gp_mqcld.Sampling import GaussianWavePacketParams, MappingInitParams, MMSTSampler

model = TullyModel(TullyParams.defaults("dual"))
dynamics = PBMEMIntDynamics(model=model, params=PBMEMIntParams(mass=2000.0, hbar=1.0))
print(model)
print("Packed dimension D =", 6)

In [ ]:
# Draw a very small initial focused sample for demonstration.
import numpy as np
rng = np.random.default_rng(0)
classical = GaussianWavePacketParams(R0=[-15.0], P0=[40.0], sigma_R=[1.0], hbar=1.0)
mapping = MappingInitParams(nstates=2, active_state=0, hbar=1.0)
sampler = MMSTSampler(classical, mapping)
s = sampler.sample_focused(n_samples=8, rng=rng)
Z0 = pack_z(s.R, s.P, s.r, s.p)
print(Z0.shape)
print("First point:", Z0[0])

## Full-density GP

The default production representation fits the complete density directly:

\[
\hat\rho(z,t)=\sum_i \alpha_i(t) k(z,Z_i(t)).
\]

The midpoint scheme updates the live empirical labels and refits the same full-density representation on the transported cloud.

## Runtime pipeline

1. sample initial cloud;
2. pack states;
3. fit initial full-density GP;
4. clone PBME and midpoint branches;
5. transport PBME branch with frozen labels;
6. transport midpoint branch, compute `Q`, update live labels, refit GP;
7. compute observables;
8. save `.npz` and `.json` files;
9. generate figures and comparison plots.